In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn. metrics import accuracy_score

In [3]:
df = pd.read_csv("loan_approved.csv")

In [4]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status (Approved)
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [5]:
X = df.drop(["Loan_Status (Approved)","Loan_ID"],axis =1 )
y = df["Loan_Status (Approved)"]

In [6]:
X_train, X_test,y_train,y_test = train_test_split(X,y,train_size = 0.75,random_state = 24)

In [7]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns

C:\Users\DELL\AppData\Local\Temp\ipykernel_22728\2931283843.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns


In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


In [9]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


In [10]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
],remainder='drop')


In [11]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)


In [12]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=15)

X_train_selected = selector.fit_transform(X_train_prep, y_train)
X_test_selected  = selector.transform(X_test_prep)


In [13]:
from sklearn.neighbors import KNeighborsClassifier

# Create KNN model
model = KNeighborsClassifier(
    n_neighbors=5,
    metric='minkowski',
    p=2  # Euclidean distance
)

# Train
model.fit(X_train_selected, y_train)

# Predict
y_pred = model.predict(X_test_selected)

In [14]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", accuracy)


Model Accuracy: 0.8571428571428571


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_classif)),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(
        n_neighbors=5,
        metric='minkowski',
        p=2  # Euclidean distance
    ))
])

In [16]:
param_grid = {
    "feature_selection__k": [10, 15, 20, "all"],
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"]
}

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)

Best Parameters: {'feature_selection__k': 10, 'model__metric': 'euclidean', 'model__n_neighbors': 11, 'model__weights': 'uniform'}
Best Accuracy: 0.7891304347826087


In [17]:
from sklearn.metrics import accuracy_score

# Best pipeline found by GridSearchCV
best_model = grid.best_estimator_

# Predict on test data
y_pred = best_model.predict(X_test)

# Calculate test accuracy
test_accuracy = accuracy_score(y_test, y_pred)



In [18]:
print("Test Accuracy:", test_accuracy)

print("Best Parameters:", grid.best_params_)
print("Cross-validation Accuracy:", grid.best_score_)
print("Test Accuracy:", accuracy_score(y_test, y_pred))


Test Accuracy: 0.8376623376623377
Best Parameters: {'feature_selection__k': 10, 'model__metric': 'euclidean', 'model__n_neighbors': 11, 'model__weights': 'uniform'}
Cross-validation Accuracy: 0.7891304347826087
Test Accuracy: 0.8376623376623377


In [19]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

# Best Parameters:
{'feature_selection__k': 15,
 'model__metric': 'euclidean',
 'model__n_neighbors': 7,
 'model__weights': 'distance'}



[[ 17  24]
 [  1 112]]
              precision    recall  f1-score   support

           N       0.94      0.41      0.58        41
           Y       0.82      0.99      0.90       113

    accuracy                           0.84       154
   macro avg       0.88      0.70      0.74       154
weighted avg       0.86      0.84      0.81       154



{'feature_selection__k': 15,
 'model__metric': 'euclidean',
 'model__n_neighbors': 7,
 'model__weights': 'distance'}

In [20]:

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_classif)),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])



y_pred = best_model.predict(X_test)



In [21]:
from sklearn.metrics import accuracy_score

# Train accuracy
y_train_pred = best_model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)

# Test accuracy
y_test_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)

print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)


Train Accuracy: 0.8021739130434783
Test Accuracy : 0.8376623376623377


In [22]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

train_accuracies = []
test_accuracies = []

for train_index, test_index in kf.split(X):
    X_train_k, X_test_k = X.iloc[train_index], X.iloc[test_index]
    y_train_k, y_test_k = y.iloc[train_index], y.iloc[test_index]

    best_model.fit(X_train_k, y_train_k)

    y_train_pred = best_model.predict(X_train_k)
    y_test_pred  = best_model.predict(X_test_k)

    train_accuracies.append(accuracy_score(y_train_k, y_train_pred))
    test_accuracies.append(accuracy_score(y_test_k, y_test_pred))


In [23]:
print("Train Accuracies:", train_accuracies)
print("Test Accuracies :", test_accuracies)

print("\nMean Train Accuracy:", np.mean(train_accuracies))
print("Mean Test Accuracy :", np.mean(test_accuracies))

print("\nTrain Accuracy Variance:", np.var(train_accuracies))
print("Test Accuracy Variance :", np.var(test_accuracies))


Train Accuracies: [0.8085539714867617, 0.7942973523421588, 0.8167006109979633, 0.8167006109979633, 0.8028455284552846]
Test Accuracies : [0.7723577235772358, 0.7642276422764228, 0.8130081300813008, 0.7886178861788617, 0.819672131147541]

Mean Train Accuracy: 0.8078196148560263
Mean Test Accuracy : 0.7915767026522724

Train Accuracy Variance: 7.317531672465902e-05
Test Accuracy Variance : 0.00047495080780607005


In [24]:
import pickle


with open("model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print("Model saved successfully as model.pkl")


Model saved successfully as model.pkl


In [26]:
import joblib

joblib.dump(best_model, "model.pkl")

['model.pkl']